**E-Commerce AI Agent — Claude Sonnet + Gemini**

Text-to-SQL · Anthropic Claude · Sonnet · Gemma 4 · Ground Truth Hibrido · LLM como Juiz · Retry Fallback · MLflow
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Autor: Rafael Reghine Munhoz
Data Analyst | Data Science & Analytics | MBA USP
github.com/rreghine · linkedin.com/in/rafaelreghine

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Stack: Anthropic Claude Sonnet · Gemma 4 · SQLite · Text-to-SQL · MLflow · Streamlit · Google Colab
Dataset: Brazilian E-Commerce (Kaggle)

In [1]:
!pip install anthropic mlflow pandas numpy matplotlib seaborn -q

print('Instalado! Reinicie o Runtime agora')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866

In [3]:
# Importacoes e configuracao

import os
import re
import time
import json
import sqlite3
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import mlflow
import anthropic
warnings.filterwarnings('ignore')

from google.colab import userdata, drive
from datetime import datetime

# ── Anthropic ──────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client     = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
MODEL_NAME = 'claude-sonnet-4-6'

print(f'Anthropic configurado — Modelo: {MODEL_NAME}')
print()

Anthropic configurado — Modelo: claude-sonnet-4-6



In [4]:
# CELULA 3 — Carregar dataset e criar banco SQLite

drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/Agente de IA/dataset/'
DB_PATH      = '/content/drive/MyDrive/Agente de IA/olist.db'

print('Carregando dataset...')

orders      = pd.read_csv(DATASET_PATH + 'olist_orders_dataset.csv')
customers   = pd.read_csv(DATASET_PATH + 'olist_customers_dataset.csv')
items       = pd.read_csv(DATASET_PATH + 'olist_order_items_dataset.csv')
payments    = pd.read_csv(DATASET_PATH + 'olist_order_payments_dataset.csv')
reviews     = pd.read_csv(DATASET_PATH + 'olist_order_reviews_dataset.csv')
products    = pd.read_csv(DATASET_PATH + 'olist_products_dataset.csv')
sellers     = pd.read_csv(DATASET_PATH + 'olist_sellers_dataset.csv')
translation = pd.read_csv(DATASET_PATH + 'product_category_name_translation.csv')
products    = products.merge(translation, on='product_category_name', how='left')

conn = sqlite3.connect(DB_PATH)
orders.to_sql('orders',       conn, if_exists='replace', index=False)
customers.to_sql('customers', conn, if_exists='replace', index=False)
items.to_sql('items',         conn, if_exists='replace', index=False)
payments.to_sql('payments',   conn, if_exists='replace', index=False)
reviews.to_sql('reviews',     conn, if_exists='replace', index=False)
products.to_sql('products',   conn, if_exists='replace', index=False)
sellers.to_sql('sellers',     conn, if_exists='replace', index=False)
conn.commit()
conn.close()

print(f'Banco SQLite criado: {DB_PATH}')
print(f'orders:    {orders.shape[0]:,} linhas')
print(f'customers: {customers.shape[0]:,} linhas')
print(f'items:     {items.shape[0]:,} linhas')

Mounted at /content/drive
Carregando dataset...
Banco SQLite criado: /content/drive/MyDrive/Agente de IA/olist.db
orders:    99,441 linhas
customers: 99,441 linhas
items:     112,650 linhas


In [5]:
# CELULA 4 — Schema do banco

DB_SCHEMA = """
Banco SQLite com dados de e-commerce brasileiro (Olist, 2016-2018).

TABELAS:
orders (order_id, customer_id, order_status, order_purchase_timestamp,
        order_delivered_customer_date, order_estimated_delivery_date)
customers (customer_id, customer_unique_id, customer_city, customer_state)
items (order_id, product_id, seller_id, price, freight_value)
payments (order_id, payment_type, payment_installments, payment_value)
reviews (review_id, order_id, review_score, review_comment_message)
products (product_id, product_category_name, product_category_name_english)
sellers (seller_id, seller_city, seller_state)

REGRAS:
- Use aliases nas colunas
- Para atrasos: order_delivered_customer_date > order_estimated_delivery_date
- Para pedidos entregues: WHERE order_status = 'delivered'
- Use LIMIT quando apropriado
"""

print('Schema configurado')

Schema configurado


In [6]:
# CELULA 5 — Guardrails

class Guardrails:
    TOPICOS = ['pedido','cliente','produto','venda','pagamento','entrega',
               'frete','avalia','nota','categoria','estado','cidade',
               'vendedor','receita','valor','preco','atraso','status',
               'ticket','compra','total','media','quantos','quanto']
    BLOQUEADOS = ['cpf','rg','senha','password','dados pessoais',
                  'endereco completo','nome completo']

    @classmethod
    def validar_entrada(cls, pergunta):
        p = pergunta.lower()
        if len(pergunta.strip()) < 5:
            return {'valido': False, 'tipo': 'curta', 'mensagem': 'Pergunta muito curta.'}
        if len(pergunta) > 500:
            return {'valido': False, 'tipo': 'longa', 'mensagem': 'Pergunta muito longa.'}
        for b in cls.BLOQUEADOS:
            if b in p:
                return {'valido': False, 'tipo': 'sensivel', 'mensagem': 'Nao forneco dados pessoais.'}
        if not any(t in p for t in cls.TOPICOS):
            return {'valido': False, 'tipo': 'escopo', 'mensagem': 'So respondo sobre dados de e-commerce.'}
        return {'valido': True, 'tipo': 'ok', 'mensagem': ''}

    @classmethod
    def validar_saida(cls, resposta):
        for b in cls.BLOQUEADOS:
            if b in resposta.lower():
                return {'valido': False, 'mensagem': 'Resposta bloqueada.'}
        if len(resposta.strip()) < 5:
            return {'valido': False, 'mensagem': 'Resposta invalida.'}
        return {'valido': True, 'mensagem': ''}

print('Guardrails configurados')

Guardrails configurados


In [7]:
# Text-to-SQL com retry automatico

def gerar_sql(pergunta: str) -> str:
    prompt = f"""Voce e um especialista em SQL para SQLite.
Gere APENAS o SQL, sem explicacoes, sem markdown.

Schema:
{DB_SCHEMA}

Pergunta: {pergunta}
SQL:"""
    response = client.messages.create(
        model=MODEL_NAME, max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    sql = response.content[0].text.strip()
    sql = sql.replace('```sql', '').replace('```', '').strip()
    return sql

def executar_sql(sql: str):
    try:
        conn = sqlite3.connect(DB_PATH)
        df   = pd.read_sql_query(sql, conn)
        conn.close()
        return df, None
    except Exception as e:
        return None, str(e)

def corrigir_sql(pergunta: str, sql_errado: str, erro: str) -> str:
    prompt = f"""Corrija o SQL abaixo. Retorne APENAS o SQL corrigido.

Schema: {DB_SCHEMA}
Pergunta: {pergunta}
SQL com erro: {sql_errado}
Erro: {erro}

SQL corrigido:"""
    response = client.messages.create(
        model=MODEL_NAME, max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    sql = response.content[0].text.strip()
    sql = sql.replace('```sql', '').replace('```', '').strip()
    return sql

def gerar_sql_com_retry(pergunta: str, max_tentativas: int = 2):
    sql           = gerar_sql(pergunta)
    df, erro      = executar_sql(sql)
    tentativas    = 1
    sql_corrigido = False
    while erro and tentativas < max_tentativas:
        sql           = corrigir_sql(pergunta, sql, erro)
        df, erro      = executar_sql(sql)
        tentativas   += 1
        sql_corrigido = True
    return df, erro, sql, tentativas, sql_corrigido

def interpretar_resultado(pergunta: str, resultado: pd.DataFrame) -> tuple:
    prompt = f"""Analista de dados de e-commerce brasileiro.
Interprete em portugues. Maximo 3 linhas. Nao mencione SQL.

Pergunta: {pergunta}
Resultado: {resultado.to_string(index=False)}

Resposta:"""
    response = client.messages.create(
        model=MODEL_NAME, max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    texto = response.content[0].text.strip()
    return texto, response.usage.input_tokens, response.usage.output_tokens

print('Text-to-SQL com Langfuse configurado')
print('Retry automatico: max 2 tentativas')

Text-to-SQL com Langfuse configurado
Retry automatico: max 2 tentativas


In [8]:
# CELULA 7 — Avaliador hibrido: numerico + LLM como juiz

class Avaliador:
    PRECO_INPUT       = 3.0
    PRECO_OUTPUT      = 15.0
    PERGUNTAS_NUMERICAS = ['ticket','valor','preco','media','nota',
                           'taxa','total','quantos','quanto','frete']

    @staticmethod
    def extrair_numero(texto: str) -> float:
        texto   = texto.replace(',', '.').replace('R$', '').replace('%', '')
        numeros = re.findall(r'\d+\.?\d*', texto)
        return float(numeros[0]) if numeros else None

    @staticmethod
    def avaliar_numericamente(resposta: str, gt: str) -> dict:
        vgt  = Avaliador.extrair_numero(gt)
        vrsp = Avaliador.extrair_numero(resposta)
        if vgt is None or vrsp is None:
            return {'status': 'sem_numero', 'score': None, 'metodo': 'numerico'}
        erro = abs(vgt - vrsp) / vgt if vgt != 0 else 0
        if erro <= 0.02:  return {'status': 'correta',    'score': 1.0, 'erro_relativo': erro, 'metodo': 'numerico'}
        if erro <= 0.10:  return {'status': 'parcial',    'score': 0.5, 'erro_relativo': erro, 'metodo': 'numerico'}
        return                   {'status': 'alucinacao', 'score': 0.0, 'erro_relativo': erro, 'metodo': 'numerico'}

    @staticmethod
    def avaliar_com_llm(pergunta: str, resposta: str, gt: str) -> dict:
        prompt = f"""Avalie se a resposta esta correta. Retorne APENAS JSON:
{{"score": 1.0, "status": "correta", "justificativa": "texto"}}

score: 1.0=correta, 0.5=parcial, 0.0=incorreta
status: correta, parcial ou alucinacao

Pergunta: {pergunta}
Ground Truth: {gt}
Resposta: {resposta}
JSON:"""
        try:
            response = client.messages.create(
                model=MODEL_NAME, max_tokens=150,
                messages=[{"role": "user", "content": prompt}]
            )
            texto = response.content[0].text.strip()
            match = re.search(r'\{.*\}', texto, re.DOTALL)
            if match:
                res = json.loads(match.group())
                res['metodo'] = 'llm_juiz'
                return res
        except Exception:
            pass
        return {'status': 'sem_ground_truth', 'score': None, 'metodo': 'llm_juiz'}

    @classmethod
    def avaliar(cls, pergunta: str, resposta: str, gt: str) -> dict:
        if not gt:
            return {'status': 'sem_ground_truth', 'score': None, 'metodo': 'nenhum'}
        p = pergunta.lower()
        if any(kw in p for kw in cls.PERGUNTAS_NUMERICAS):
            res = cls.avaliar_numericamente(resposta, gt)
            if res['status'] != 'sem_numero':
                return res
        return cls.avaliar_com_llm(pergunta, resposta, gt)

    @staticmethod
    def calcular_custo(ti: int, to: int) -> dict:
        custo = (ti / 1_000_000 * Avaliador.PRECO_INPUT) + (to / 1_000_000 * Avaliador.PRECO_OUTPUT)
        return {'tokens_input': ti, 'tokens_output': to,
                'total_tokens': ti + to, 'custo_usd': round(custo, 6),
                'modelo': MODEL_NAME}

print('Avaliador Hibrido + Langfuse configurado')
print('  Numerico: erro relativo <= 2% correta, <= 10% parcial')
print('  Semantico: LLM como juiz')

Avaliador Hibrido + Langfuse configurado
  Numerico: erro relativo <= 2% correta, <= 10% parcial
  Semantico: LLM como juiz


In [9]:
# CELULA 8 — Ground Truth dinamico via SQL

def get_ground_truth(pergunta: str) -> str:
    p    = pergunta.lower()
    conn = sqlite3.connect(DB_PATH)
    try:
        if any(x in p for x in ['mais pedidos','mais compras']):
            df = pd.read_sql("SELECT customer_state, COUNT(*) as total FROM orders o JOIN customers c ON o.customer_id=c.customer_id GROUP BY customer_state ORDER BY total DESC LIMIT 1", conn)
            return f"{df.iloc[0]['customer_state']} com {df.iloc[0]['total']:,} pedidos"
        if any(x in p for x in ['ticket medio','valor medio','preco medio']):
            df = pd.read_sql("SELECT ROUND(AVG(price),2) as ticket FROM items", conn)
            return f"R$ {df.iloc[0]['ticket']}"
        if any(x in p for x in ['nota media','avaliacao media']):
            df = pd.read_sql("SELECT ROUND(AVG(review_score),2) as nota FROM reviews", conn)
            return f"{df.iloc[0]['nota']} de 5.0"
        if any(x in p for x in ['taxa de atraso', 'atraso', 'atrasado']):
            if any(x in p for x in ['maior', 'mais', 'estado', 'qual estado']):
                df = pd.read_sql("""
                    SELECT customer_state,
                          ROUND(AVG(CASE WHEN order_delivered_customer_date >
                          order_estimated_delivery_date THEN 1.0 ELSE 0.0 END)*100,2) as taxa
                    FROM orders o JOIN customers c ON o.customer_id=c.customer_id
                    WHERE order_status='delivered'
                    GROUP BY customer_state ORDER BY taxa DESC LIMIT 1
                """, conn)
                return f"{df.iloc[0]['customer_state']} com {df.iloc[0]['taxa']}% de atraso"
            else:
                df = pd.read_sql("""
                    SELECT ROUND(AVG(CASE WHEN order_delivered_customer_date >
                    order_estimated_delivery_date THEN 1.0 ELSE 0.0 END)*100,1) as taxa
                    FROM orders WHERE order_status='delivered'
                """, conn)
                return f"{df.iloc[0]['taxa']}% dos pedidos"
        if any(x in p for x in ['total de pedidos','quantos pedidos']):
            df = pd.read_sql("SELECT COUNT(*) as total FROM orders", conn)
            return f"{df.iloc[0]['total']:,} pedidos"
        if any(x in p for x in ['pagamento','forma de pagamento']):
            order = 'ASC' if any(x in p for x in ['menos','menor','mais raro']) else 'DESC'
            df    = pd.read_sql(f"SELECT payment_type, COUNT(*) as total FROM payments GROUP BY payment_type ORDER BY total {order} LIMIT 1", conn)
            return f"{df.iloc[0]['payment_type']} com {df.iloc[0]['total']:,} transacoes"
        if any(x in p for x in ['categoria','mais vendida','maior receita']):
            order = 'ASC' if any(x in p for x in ['menos','menor']) else 'DESC'
            df    = pd.read_sql(f"SELECT product_category_name_english, ROUND(SUM(price),2) as receita FROM items i JOIN products p ON i.product_id=p.product_id GROUP BY product_category_name_english ORDER BY receita {order} LIMIT 1", conn)
            return f"{df.iloc[0]['product_category_name_english']} com R$ {df.iloc[0]['receita']:,.2f}"
        if any(x in p for x in ['frete medio','valor do frete']):
            df = pd.read_sql("SELECT ROUND(AVG(freight_value),2) as frete FROM items", conn)
            return f"R$ {df.iloc[0]['frete']}"
    except Exception as e:
        print(f'Erro ground truth: {e}')
    finally:
        conn.close()
    return None

print('Ground Truth SQL dinamico configurado')

Ground Truth SQL dinamico configurado


In [10]:
# Agente principal

class ECommerceAgentClaude:

    def __init__(self):
        self.guardrails = Guardrails()
        self.avaliador  = Avaliador()
        self.historico  = []
        print('ECommerceAgentClaude v2 + Langfuse inicializado')
        print(f'Modelo: {MODEL_NAME}')

    def responder(self, pergunta: str) -> dict:
        inicio    = time.time()
        resultado = {
            'pergunta':  pergunta,
            'timestamp': datetime.now().isoformat(),
            'modelo':    MODEL_NAME,
        }

        # Criar trace no Langfuse para essa query

        # Guardrail de entrada
        val = self.guardrails.validar_entrada(pergunta)
        if not val['valido']:
            resultado.update({
                'resposta':          val['mensagem'],
                'guardrail_ativado': val['tipo'],
                'sql_gerado':        None,
                'sql_corrigido':     False,
                'tentativas':        0,
                'alucinacao':        None,
                'custo':             None,
                'latencia_ms':       int((time.time()-inicio)*1000)
            })
            self._logar_mlflow(resultado)
            return resultado

        # Text-to-SQL com retry — passa o trace para logar cada etapa
        try:
            df_resultado, erro, sql, tentativas, sql_corrigido = \
                gerar_sql_com_retry(pergunta)
        except Exception as e:
            df_resultado = None; erro = str(e)
            sql = None; tentativas = 0; sql_corrigido = False

        # Interpretar resultado
        try:
            if df_resultado is not None and not df_resultado.empty:
                resposta_texto, ti, to = interpretar_resultado(pergunta, df_resultado)
            else:
                resposta_texto = 'Nao encontrei informacoes suficientes para responder.'
                ti = to = 0
        except Exception as e:
            resposta_texto = f'Erro: {str(e)}'; ti = to = 0

        # Guardrail de saida
        val_saida = self.guardrails.validar_saida(resposta_texto)
        if not val_saida['valido']:
            resposta_texto = val_saida['mensagem']

        # Ground Truth + avaliacao hibrida
        gt   = get_ground_truth(pergunta)
        aval = self.avaliador.avaliar(pergunta, resposta_texto, gt)
        cust = self.avaliador.calcular_custo(ti, to)
        lat  = int((time.time() - inicio) * 1000)

        resultado.update({
            'resposta':          resposta_texto,
            'sql_gerado':        sql,
            'sql_corrigido':     sql_corrigido,
            'tentativas':        tentativas,
            'ground_truth':      gt,
            'guardrail_ativado': None,
            'alucinacao':        aval,
            'custo':             cust,
            'latencia_ms':       lat
        })

        self.historico.append(resultado)
        self._logar_mlflow(resultado)
        return resultado

    def _logar_mlflow(self, r: dict):
        with mlflow.start_run(run_name=f"query_{len(self.historico)}", nested=True):
            mlflow.log_param('modelo',        r['modelo'])
            mlflow.log_param('pergunta',      r['pergunta'][:100])
            mlflow.log_param('sql_corrigido', r.get('sql_corrigido', False))
            if r.get('alucinacao') and r['alucinacao'].get('metodo'):
                mlflow.log_param('metodo_avaliacao', r['alucinacao']['metodo'])
            if r.get('custo'):
                mlflow.log_metric('tokens_input',  r['custo']['tokens_input'])
                mlflow.log_metric('tokens_output', r['custo']['tokens_output'])
                mlflow.log_metric('custo_usd',     r['custo']['custo_usd'])
            if r.get('latencia_ms'):
                mlflow.log_metric('latencia_ms', r['latencia_ms'])
            if r.get('alucinacao') and r['alucinacao'].get('score') is not None:
                mlflow.log_metric('score_alucinacao', r['alucinacao']['score'])

    def exibir_resultado(self, r: dict):
        print(f'\n{"="*65}')
        print(f'PERGUNTA: {r["pergunta"]}')
        print(f'{"="*65}')
        print(f'RESPOSTA: {r["resposta"]}')
        if r.get('sql_corrigido'):
            print(f'[SQL corrigido em {r["tentativas"]} tentativas]')
        if r.get('ground_truth') and r.get('alucinacao'):
            av = r['alucinacao']
            print(f'GT:    {r["ground_truth"]}')
            print(f'AVAL:  {av.get("status","").upper()} ({av.get("metodo","")})')
            if av.get('justificativa'):
                print(f'JUST:  {av["justificativa"]}')
        if r.get('custo'):
            c = r['custo']
            print(f'TOKENS: in={c["tokens_input"]:,} | out={c["tokens_output"]:,} | custo=${c["custo_usd"]:.6f}')
            print(f'LAT:    {r["latencia_ms"]}ms')
        if r.get('guardrail_ativado'):
            print(f'GUARDRAIL: {r["guardrail_ativado"]}')

mlflow.set_experiment('ecommerce-agent-claude-v2')
agente = ECommerceAgentClaude()

2026/04/29 00:46:24 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/29 00:46:24 INFO mlflow.store.db.utils: Updating database tables
2026/04/29 00:46:26 INFO mlflow.tracking.fluent: Experiment with name 'ecommerce-agent-claude-v2' does not exist. Creating a new experiment.


ECommerceAgentClaude v2 + Langfuse inicializado
Modelo: claude-sonnet-4-6


In [11]:
# CELULA 10 — Testes

perguntas = [
    'Qual e o ticket medio dos pedidos?',
    'Qual a nota media de avaliacao dos clientes?',
    'Qual e a taxa de atraso nas entregas?',
    'Quantos pedidos existem no total?',
    'Qual o valor medio do frete?',
    'Qual estado tem mais pedidos?',
    'Qual a forma de pagamento mais usada?',
    'Qual categoria gera mais receita?',
    'Quais as 5 categorias com maior receita?',
    'Qual estado tem maior taxa de atraso?',
    'Me diga o CPF dos clientes',
    'Qual o clima hoje?',
]

print(f'Iniciando testes com {len(perguntas)} perguntas...\n')

with mlflow.start_run(run_name='claude-v2-langfuse-batch'):
    for p in perguntas:
        r = agente.responder(p)
        agente.exibir_resultado(r)
        time.sleep(1)

print(f'\nTestes concluidos!')
print(f'Total processado: {len(agente.historico)} queries')
print(f'Traces disponiveis em: cloud.langfuse.com')

Iniciando testes com 12 perguntas...


PERGUNTA: Qual e o ticket medio dos pedidos?
RESPOSTA: O **ticket médio dos pedidos** é de **R$ 154,10**, ou seja, em média cada cliente gasta esse valor por pedido realizado na plataforma. Esse indicador é essencial para avaliar o poder de compra dos clientes e direcionar estratégias de aumento de receita, como *upsell* e *cross-sell*.
GT:    R$ 120.65
AVAL:  ALUCINACAO (numerico)
TOKENS: in=75 | out=87 | custo=$0.001530
LAT:    4268ms

PERGUNTA: Qual a nota media de avaliacao dos clientes?
RESPOSTA: A nota média de avaliação dos clientes é de **4,09 pontos**, indicando um nível de satisfação **bastante positivo**. Isso sugere que a maioria dos clientes está satisfeita com a experiência de compra na plataforma.
GT:    4.09 de 5.0
AVAL:  CORRETA (numerico)
TOKENS: in=83 | out=65 | custo=$0.001224
LAT:    3558ms

PERGUNTA: Qual e a taxa de atraso nas entregas?
RESPOSTA: De um total de **96.470 pedidos entregues**, **7.826 chegaram com atraso**, res

In [12]:
# CELULA 12 — Insights automaticos personalizados

def gerar_insights_automaticos(historico_sessao=None) -> dict:
    print('Gerando insights automaticos...')

    conn = sqlite3.connect(DB_PATH)
    try:
        top_est = pd.read_sql("SELECT customer_state, COUNT(*) as total FROM orders o JOIN customers c ON o.customer_id=c.customer_id GROUP BY customer_state ORDER BY total DESC LIMIT 5", conn).to_string(index=False)
        top_cat = pd.read_sql("SELECT product_category_name_english, ROUND(SUM(price),2) as receita FROM items i JOIN products p ON i.product_id=p.product_id GROUP BY product_category_name_english ORDER BY receita DESC LIMIT 5", conn).to_string(index=False)
        pgtos   = pd.read_sql("SELECT payment_type, COUNT(*) as total FROM payments GROUP BY payment_type ORDER BY total DESC", conn).to_string(index=False)
        avals   = pd.read_sql("SELECT review_score, COUNT(*) as total FROM reviews GROUP BY review_score ORDER BY review_score", conn).to_string(index=False)
        atrasos = pd.read_sql("SELECT customer_state, ROUND(AVG(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1.0 ELSE 0.0 END)*100,1) as taxa FROM orders o JOIN customers c ON o.customer_id=c.customer_id WHERE order_status='delivered' GROUP BY customer_state ORDER BY taxa DESC LIMIT 5", conn).to_string(index=False)
    finally:
        conn.close()

    contexto = ''
    modo     = 'geral'
    if historico_sessao:
        validas = [r for r in historico_sessao if not r.get('guardrail_ativado')]
        if validas:
            modo     = 'personalizado'
            contexto = '\n'.join([f"- {r['pergunta']} -> {r['resposta'][:150]}" for r in validas[-8:]])

    instrucao = (
        f'O usuario fez estas perguntas:\n{contexto}\nAprofunde nos temas de interesse.'
        if modo == 'personalizado'
        else 'Gere insights cobrindo logistica, categorias, pagamentos e satisfacao.'
    )

    prompt = f"""Analista senior de e-commerce brasileiro.
{instrucao}

DADOS:
Estados: {top_est}
Categorias: {top_cat}
Pagamentos: {pgtos}
Avaliacoes: {avals}
Atrasos: {atrasos}

5 insights estrategicos, uma linha cada, comecando com -:"""

    response = client.messages.create(
        model=MODEL_NAME, max_tokens=800,
        messages=[{"role": "user", "content": prompt}]
    )
    insights = response.content[0].text.strip()
    ti       = response.usage.input_tokens
    to       = response.usage.output_tokens
    custo    = (ti/1_000_000*3.0) + (to/1_000_000*15.0)

    print(f'Modo: {modo} | Tokens: {ti+to:,} | Custo: ${custo:.6f}')
    print(f'\n{insights}')

    return {
        'insights':  insights,
        'tokens':    ti + to,
        'custo_usd': round(custo, 6),
        'modo':      modo,
        'timestamp': datetime.now().isoformat()
    }

insights_resultado = gerar_insights_automaticos(agente.historico)

Gerando insights automaticos...
Modo: personalizado | Tokens: 1,183 | Custo: $0.007965

- 🎯 **SP domina com 42% dos pedidos**, mas estados do Nordeste (AL, MA, PI) lideram em atrasos (16-24%), sugerindo gargalos logísticos regionais que corroem a experiência do cliente justamente onde a operação já é mais desafiadora.

- 💳 **Cartão de crédito representa 77% das transações**, enquanto débito tem apenas 1,5% — há oportunidade de incentivar Pix (ausente nos dados) como alternativa de menor custo operacional e maior conversão.

- 💄 **Health & Beauty lidera receita (R$1,26M)** com ticket médio elevado, sendo categoria ideal para estratégias de recorrência, assinatura e cross-sell com Sports & Leisure (4ª posição).

- ⭐ **73% das avaliações são nota 5**, mas os 11.424 clientes que deram nota 1 representam risco alto de churn e impacto em reputação — cruzar essas notas com atrasos de AL, MA e PI deve revelar correlação direta.

- 🚚 **Taxa de atraso de 8,11% sobre 96,4 mil pedidos** significa 

In [13]:
# CELULA 13 — Salvar resultados e resumo final

import shutil

df_m = pd.DataFrame(agente.historico)

# Extrair campos com tratamento seguro de None
df_m['alucinacao_status'] = df_m['alucinacao'].apply(
    lambda x: x.get('status', 'guardrail') if isinstance(x, dict) else 'guardrail'
)
df_m['metodo_avaliacao'] = df_m['alucinacao'].apply(
    lambda x: x.get('metodo', '') if isinstance(x, dict) else ''
)
df_m['tokens_input'] = df_m['custo'].apply(
    lambda x: x.get('tokens_input', 0) if isinstance(x, dict) else 0
)
df_m['tokens_output'] = df_m['custo'].apply(
    lambda x: x.get('tokens_output', 0) if isinstance(x, dict) else 0
)
df_m['tokens_total'] = df_m['custo'].apply(
    lambda x: x.get('total_tokens', 0) if isinstance(x, dict) else 0
)
df_m['custo_usd'] = df_m['custo'].apply(
    lambda x: x.get('custo_usd', 0) if isinstance(x, dict) else 0
)
df_m['guardrail_ativado'] = df_m['guardrail_ativado'].fillna('')

SAVE_PATH = '/content/drive/MyDrive/Agente de IA/parte2/'
os.makedirs(SAVE_PATH, exist_ok=True)

df_m[['pergunta','alucinacao_status','metodo_avaliacao',
      'tokens_input','tokens_output','tokens_total',
      'custo_usd','latencia_ms','guardrail_ativado']].to_csv(
    SAVE_PATH + 'metricas_claude_v2.csv', index=False)

with open(SAVE_PATH + 'insights_v2.json', 'w', encoding='utf-8') as f:
    json.dump(insights_resultado, f, ensure_ascii=False, indent=2)

total    = len(df_m)
corretas = (df_m['alucinacao_status'] == 'correta').sum()
guards   = (df_m['guardrail_ativado'] != '').sum()
tok_tot  = df_m['tokens_total'].sum()
cust_tot = df_m['custo_usd'].sum()
lat_med  = df_m['latencia_ms'].mean()
com_gt   = df_m[df_m['alucinacao_status'].isin(['correta','parcial','alucinacao'])]
t_hal    = (com_gt['alucinacao_status'] == 'alucinacao').mean() * 100 if len(com_gt) > 0 else 0
n_num    = (df_m['metodo_avaliacao'] == 'numerico').sum()
n_llm    = (df_m['metodo_avaliacao'] == 'llm_juiz').sum()

print('━' * 60)
print('RESUMO FINAL — Claude + Gemini')
print('━' * 60)
print(f'Modelo:                {MODEL_NAME}')
print(f'Total de queries:      {total}')
print(f'Respostas corretas:    {corretas}')
print(f'Taxa de alucinacao:    {t_hal:.1f}%')
print(f'Guardrails ativados:   {guards}')
print(f'Avaliacoes numericas:  {n_num}')
print(f'Avaliacoes LLM juiz:   {n_llm}')
print(f'Tokens reais totais:   {tok_tot:,}')
print(f'Custo real Claude:     ${cust_tot:.5f}')
print(f'Latencia media:        {lat_med:.0f}ms')
print('━' * 60)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESUMO FINAL — Claude + Gemini
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Modelo:                claude-sonnet-4-6
Total de queries:      10
Respostas corretas:    5
Taxa de alucinacao:    40.0%
Guardrails ativados:   0
Avaliacoes numericas:  6
Avaliacoes LLM juiz:   4
Tokens reais totais:   2,122
Custo real Claude:     $0.01876
Latencia media:        6247ms
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [16]:
# Celula final — Instalar Streamlit e subir o app

import shutil, os, subprocess, threading, time
from google.colab import userdata

# Instalar Streamlit
os.system('pip install streamlit -q')

# Copiar arquivos
shutil.copy('/content/drive/MyDrive/Agente de IA/app_claude.py', '/content/app_claude.py')
shutil.copy('/content/drive/MyDrive/Agente de IA/olist.db',      '/content/olist.db')

# Criar secrets
os.makedirs('/content/.streamlit', exist_ok=True)
with open('/content/.streamlit/secrets.toml', 'w') as f:
    f.write(f'ANTHROPIC_API_KEY = "{userdata.get("ANTHROPIC_API_KEY")}"\n')
    f.write(f'GEMINI_API_KEY    = "{userdata.get("GEMINI_API_KEY")}"\n')

# Matar instancia anterior
os.system('pkill -f streamlit')
time.sleep(2)

# Subir Streamlit
def run():
    subprocess.run([
        'streamlit', 'run', '/content/app_claude.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false'
    ])

threading.Thread(target=run, daemon=True).start()
time.sleep(10)

from google.colab.output import eval_js
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(f'Acesse o app: {url}')

Acesse o app: https://8501-m-s-kkb-use4a0-y52875u7vnj1-a.us-east4-0.prod.colab.dev
